In [1]:
import pandas as pd
import numpy as np

In [2]:
products = pd.read_csv(
    "../data/processed/products_clean.csv"
)

reviews = pd.read_csv(
    "../data/processed/skincare_reviews_clean.csv",
    parse_dates=["submission_time"]
)

/var/folders/gw/b8128pz11k31ky_873j0sxsc0000gn/T/ipykernel_56201/2508245933.py:5: DtypeWarning: Columns (0: author_id) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv(


In [10]:
negative_reviews = reviews[
    reviews["rating"] <= 2
].copy()

In [12]:
complaint_themes = {
    "dryness": [
        "dry", "drying", "dried", "dehydrated"
    ],
    "irritation": [
        "irritated", "irritation", "burning", "burned", "sting", "stinging"
    ],
    "breakouts": [
        "breakout", "breakouts", "broke out", "acne", "pimples"
    ],
    "ineffective": [
        "didn work", "doesn work", "no difference", "not effective"
    ],
    "value": [
        "waste money", "not worth", "too expensive", "overpriced"
    ],
    "texture": [
        "sticky", "greasy", "oily", "heavy", "thick"
    ],
    "smell": [
        "smell", "scent", "fragrance"
    ],
    "packaging": [
        "pump", "bottle", "packaging", "container", "leak"
    ]
}

In [13]:
def count_theme_mentions(df, text_column, themes):
    results = []

    text = df[text_column].str.lower()

    for theme, keywords in themes.items():
        pattern = "|".join(keywords)

        mentions = text.str.contains(
            pattern,
            regex=True,
            na=False
        )

        count = mentions.sum()
        percent = (count / len(df)) * 100

        results.append({
            "theme": theme,
            "review_count": count,
            "percent_of_reviews": round(percent, 2)
        })

    return pd.DataFrame(results).sort_values(
        "percent_of_reviews",
        ascending=False
    )

In [14]:
skincare_baseline = count_theme_mentions(
    negative_reviews,
    "review_text",
    complaint_themes
)

skincare_baseline

,theme,review_count,percent_of_reviews
6,smell,24666,21.63
5,texture,24255,21.26
0,dryness,24196,21.21
2,breakouts,19606,17.19
7,packaging,11585,10.16
1,irritation,9122,8.00
4,value,6005,5.26
3,ineffective,1992,1.75


In [15]:
product_name = "Caffeine 5% + EGCG Depuffing Eye Serum"

product_reviews = reviews[
    reviews["product_name"] == product_name
].copy()

product_negative = product_reviews[
    product_reviews["rating"] <= 2
].copy()

print("Total reviews:", len(product_reviews))
print("Negative reviews:", len(product_negative))

Total reviews: 2114
Negative reviews: 490


In [16]:
product_complaints = count_theme_mentions(
    product_negative,
    "review_text",
    complaint_themes
)

product_complaints

,theme,review_count,percent_of_reviews
0,dryness,97,19.80
7,packaging,45,9.18
5,texture,38,7.76
1,irritation,37,7.55
3,ineffective,32,6.53
4,value,9,1.84
2,breakouts,6,1.22
6,smell,4,0.82


In [17]:
diagnostic = product_complaints.merge(
    skincare_baseline[["theme", "percent_of_reviews"]],
    on="theme",
    suffixes=("_product", "_baseline")
)

diagnostic["lift"] = (
    diagnostic["percent_of_reviews_product"] /
    diagnostic["percent_of_reviews_baseline"]
).round(2)

diagnostic = diagnostic.sort_values(
    "lift",
    ascending=False
)

diagnostic

,theme,review_count,percent_of_reviews_product,percent_of_reviews_baseline,lift
4,ineffective,32,6.53,1.75,3.73
3,irritation,37,7.55,8.00,0.94
0,dryness,97,19.80,21.21,0.93
1,packaging,45,9.18,10.16,0.90
2,texture,38,7.76,21.26,0.37
5,value,9,1.84,5.26,0.35
6,breakouts,6,1.22,17.19,0.07
7,smell,4,0.82,21.63,0.04


In [18]:
def build_product_diagnostic(product_name, reviews, complaint_themes, baseline):
    product_reviews = reviews[
        reviews["product_name"] == product_name
    ].copy()

    product_negative = product_reviews[
        product_reviews["rating"] <= 2
    ].copy()

    product_complaints = count_theme_mentions(
        product_negative,
        "review_text",
        complaint_themes
    )

    diagnostic = product_complaints.merge(
        baseline[["theme", "percent_of_reviews"]],
        on="theme",
        suffixes=("_product", "_baseline")
    )

    diagnostic["lift"] = (
        diagnostic["percent_of_reviews_product"] /
        diagnostic["percent_of_reviews_baseline"]
    ).round(2)

    diagnostic["product_name"] = product_name
    diagnostic["total_reviews"] = len(product_reviews)
    diagnostic["negative_reviews"] = len(product_negative)

    return diagnostic.sort_values(
        "lift",
        ascending=False
    )

In [20]:
def summarize_diagnostic(diagnostic, min_count=20, min_lift=1.5):
    meaningful = diagnostic[
        (diagnostic["review_count"] >= min_count) &
        (diagnostic["lift"] >= min_lift)
    ].copy()

    return meaningful.sort_values(
        "lift",
        ascending=False
    )

In [21]:
meaningful_issues = summarize_diagnostic(
    test_diagnostic
)

meaningful_issues

,theme,review_count,percent_of_reviews_product,percent_of_reviews_baseline,lift,product_name,total_reviews,negative_reviews
0,dryness,208,40.31,21.21,1.90,Beste No. 9 Jelly Cleanser,2692,516
1,breakouts,167,32.36,17.19,1.88,Beste No. 9 Jelly Cleanser,2692,516


In [22]:
test_diagnostic = build_product_diagnostic(
    "Beste No. 9 Jelly Cleanser",
    reviews,
    complaint_themes,
    skincare_baseline
)

test_diagnostic

,theme,review_count,percent_of_reviews_product,percent_of_reviews_baseline,lift,product_name,total_reviews,negative_reviews
0,dryness,208,40.31,21.21,1.90,Beste No. 9 Jelly Cleanser,2692,516
1,breakouts,167,32.36,17.19,1.88,Beste No. 9 Jelly Cleanser,2692,516
3,irritation,57,11.05,8.00,1.38,Beste No. 9 Jelly Cleanser,2692,516
6,value,23,4.46,5.26,0.85,Beste No. 9 Jelly Cleanser,2692,516
2,texture,62,12.02,21.26,0.57,Beste No. 9 Jelly Cleanser,2692,516
5,packaging,25,4.84,10.16,0.48,Beste No. 9 Jelly Cleanser,2692,516
4,smell,51,9.88,21.63,0.46,Beste No. 9 Jelly Cleanser,2692,516
7,ineffective,1,0.19,1.75,0.11,Beste No. 9 Jelly Cleanser,2692,516


In [23]:
candidate_products = products[
    (products["primary_category"] == "Skincare") &
    (products["rating"].notna()) &
    (products["rating"] < 4.1) &
    (products["reviews"] >= 300) &
    (products["loves_count"] >= 10000)
].copy()

candidate_products.shape

(175, 16)

In [24]:
all_diagnostics = []

for product_name in candidate_products["product_name"].unique():
    diagnostic = build_product_diagnostic(
        product_name,
        reviews,
        complaint_themes,
        skincare_baseline
    )

    meaningful = summarize_diagnostic(diagnostic)

    if not meaningful.empty:
        all_diagnostics.append(meaningful)

In [25]:
all_diagnostics_df = pd.concat(
    all_diagnostics,
    ignore_index=True
)

all_diagnostics_df.shape

(186, 8)

In [26]:
all_diagnostics_df.sort_values(
    "lift",
    ascending=False
).head(30)

,theme,review_count,percent_of_reviews_product,percent_of_reviews_baseline,lift,product_name,total_reviews,negative_reviews
162,packaging,116,78.91,10.16,7.77,The Silk Sunscreen Mineral Broad Spectrum SPF ...,485,147
26,ineffective,26,11.11,1.75,6.35,Focuspot Micro Tip Patches,577,234
15,ineffective,21,9.21,1.75,5.26,Pep-Start Eye Cream,1005,228
130,packaging,24,46.15,10.16,4.54,Super Amino Gel Cleanser,313,52
105,ineffective,23,7.17,1.75,4.10,Oil-Absorbing Pore Treatment Strips,1566,321
68,packaging,53,41.41,10.16,4.08,"Glow Clear, Color Correcting Self-Tanning Mousse",392,128
18,breakouts,75,66.37,17.19,3.86,Acne Solutions All-Over Clearing Treatment Oil...,712,113
172,ineffective,32,6.53,1.75,3.73,Caffeine 5% + EGCG Depuffing Eye Serum,2114,490
91,breakouts,40,63.49,17.19,3.69,Drying Lotion,396,63
184,smell,52,78.79,21.63,3.64,The Cult Classic Purifying Face Cleanser,410,66


In [27]:
top_issue_per_product = (
    all_diagnostics_df
    .sort_values("lift", ascending=False)
    .groupby("product_name", as_index=False)
    .first()
)

top_issue_per_product.head(20)

,product_name,theme,review_count,percent_of_reviews_product,percent_of_reviews_baseline,lift,total_reviews,negative_reviews
0,(Re) Setting Refreshing Mist SPF 40,packaging,28,30.43,10.16,3.00,382,92
1,(Re)setting 100% Mineral Powder Sunscreen SPF ...,packaging,21,33.87,10.16,3.33,377,62
2,15% Vitamin C and EGF Brightening Serum,packaging,40,36.04,10.16,3.55,671,111
3,24K Gold Pure Luxury Lift & Firm Hydra-Gel Eye...,dryness,33,38.37,21.21,1.81,321,86
4,Acne Solutions All-Over Clearing Treatment Oil...,breakouts,75,66.37,17.19,3.86,712,113
5,Acne-Clear Invisible Dots,breakouts,84,44.68,17.19,2.60,554,188
6,Alpha Arbutin 2% + HA Hyperpigmentation Serum,packaging,25,18.94,10.16,1.86,755,132
7,Argan Cleansing Oil,texture,171,49.71,21.26,2.34,2022,344
8,Argan Daily Moisturizer Tinted SPF 47 Protect ...,texture,35,42.68,21.26,2.01,488,82
9,Ascorbic Acid 8% + Alpha Arbutin 2%,texture,54,45.00,21.26,2.12,499,120


In [28]:
diagnostic_summary = top_issue_per_product.merge(
    products[
        [
            "product_name",
            "brand_name",
            "price_usd",
            "rating",
            "reviews",
            "loves_count"
        ]
    ],
    on="product_name",
    how="left"
)

In [29]:
diagnostic_summary = diagnostic_summary[
    [
        "product_name",
        "brand_name",
        "price_usd",
        "rating",
        "reviews",
        "loves_count",
        "theme",
        "review_count",
        "percent_of_reviews_product",
        "percent_of_reviews_baseline",
        "lift"
    ]
].sort_values(
    "lift",
    ascending=False
)

diagnostic_summary.head(20)

,product_name,brand_name,price_usd,rating,reviews,loves_count,theme,review_count,percent_of_reviews_product,percent_of_reviews_baseline,lift
113,The Silk Sunscreen Mineral Broad Spectrum SPF ...,Tatcha,62.00,3.3975,483.0,37047,packaging,116,78.91,10.16,7.77
44,Focuspot Micro Tip Patches,Dr. Jart+,20.00,3.0381,578.0,54955,ineffective,26,11.11,1.75,6.35
84,Pep-Start Eye Cream,CLINIQUE,32.00,3.6769,1006.0,47677,ineffective,21,9.21,1.75,5.26
105,Super Amino Gel Cleanser,Summer Fridays,35.00,4.0833,312.0,30871,packaging,24,46.15,10.16,4.54
81,Oil-Absorbing Pore Treatment Strips,Peace Out,19.00,3.9750,1562.0,102327,ineffective,23,7.17,1.75,4.10
46,"Glow Clear, Color Correcting Self-Tanning Mousse",Isle of Paradise,32.00,3.4289,394.0,23933,packaging,53,41.41,10.16,4.08
4,Acne Solutions All-Over Clearing Treatment Oil...,CLINIQUE,26.00,4.0182,713.0,17157,breakouts,75,66.37,17.19,3.86
21,Caffeine 5% + EGCG Depuffing Eye Serum,The Ordinary,8.90,3.7715,2118.0,281928,ineffective,32,6.53,1.75,3.73
39,Drying Lotion,Mario Badescu,17.00,4.0955,398.0,57039,breakouts,40,63.49,17.19,3.69
111,The Cult Classic Purifying Face Cleanser,TULA Skincare,34.00,3.9064,406.0,11798,smell,52,78.79,21.63,3.64


In [30]:
def classify_issue(row):
    if row["lift"] >= 5:
        return "Very strong abnormal issue"
    elif row["lift"] >= 3:
        return "Strong abnormal issue"
    elif row["lift"] >= 2:
        return "Moderate abnormal issue"
    else:
        return "Elevated issue"

In [31]:
diagnostic_summary["issue_strength"] = diagnostic_summary.apply(
    classify_issue,
    axis=1
)

diagnostic_summary.head(20)

,product_name,brand_name,price_usd,rating,reviews,loves_count,theme,review_count,percent_of_reviews_product,percent_of_reviews_baseline,lift,issue_strength
113,The Silk Sunscreen Mineral Broad Spectrum SPF ...,Tatcha,62.00,3.3975,483.0,37047,packaging,116,78.91,10.16,7.77,Very strong abnormal issue
44,Focuspot Micro Tip Patches,Dr. Jart+,20.00,3.0381,578.0,54955,ineffective,26,11.11,1.75,6.35,Very strong abnormal issue
84,Pep-Start Eye Cream,CLINIQUE,32.00,3.6769,1006.0,47677,ineffective,21,9.21,1.75,5.26,Very strong abnormal issue
105,Super Amino Gel Cleanser,Summer Fridays,35.00,4.0833,312.0,30871,packaging,24,46.15,10.16,4.54,Strong abnormal issue
81,Oil-Absorbing Pore Treatment Strips,Peace Out,19.00,3.9750,1562.0,102327,ineffective,23,7.17,1.75,4.10,Strong abnormal issue
46,"Glow Clear, Color Correcting Self-Tanning Mousse",Isle of Paradise,32.00,3.4289,394.0,23933,packaging,53,41.41,10.16,4.08,Strong abnormal issue
4,Acne Solutions All-Over Clearing Treatment Oil...,CLINIQUE,26.00,4.0182,713.0,17157,breakouts,75,66.37,17.19,3.86,Strong abnormal issue
21,Caffeine 5% + EGCG Depuffing Eye Serum,The Ordinary,8.90,3.7715,2118.0,281928,ineffective,32,6.53,1.75,3.73,Strong abnormal issue
39,Drying Lotion,Mario Badescu,17.00,4.0955,398.0,57039,breakouts,40,63.49,17.19,3.69,Strong abnormal issue
111,The Cult Classic Purifying Face Cleanser,TULA Skincare,34.00,3.9064,406.0,11798,smell,52,78.79,21.63,3.64,Strong abnormal issue


In [32]:
diagnostic_summary["insight"] = diagnostic_summary.apply(
    lambda row:
        f"{row['theme'].title()} complaints appear "
        f"{row['lift']:.1f}× more often than the skincare baseline "
        f"({row['percent_of_reviews_product']:.1f}% vs "
        f"{row['percent_of_reviews_baseline']:.1f}%).",
    axis=1
)

In [33]:
diagnostic_summary[
    [
        "product_name",
        "brand_name",
        "rating",
        "reviews",
        "theme",
        "lift",
        "issue_strength",
        "insight"
    ]
].head(20)

,product_name,brand_name,rating,reviews,theme,lift,issue_strength,insight
113,The Silk Sunscreen Mineral Broad Spectrum SPF ...,Tatcha,3.3975,483.0,packaging,7.77,Very strong abnormal issue,Packaging complaints appear 7.8× more often th...
44,Focuspot Micro Tip Patches,Dr. Jart+,3.0381,578.0,ineffective,6.35,Very strong abnormal issue,Ineffective complaints appear 6.3× more often ...
84,Pep-Start Eye Cream,CLINIQUE,3.6769,1006.0,ineffective,5.26,Very strong abnormal issue,Ineffective complaints appear 5.3× more often ...
105,Super Amino Gel Cleanser,Summer Fridays,4.0833,312.0,packaging,4.54,Strong abnormal issue,Packaging complaints appear 4.5× more often th...
81,Oil-Absorbing Pore Treatment Strips,Peace Out,3.9750,1562.0,ineffective,4.10,Strong abnormal issue,Ineffective complaints appear 4.1× more often ...
46,"Glow Clear, Color Correcting Self-Tanning Mousse",Isle of Paradise,3.4289,394.0,packaging,4.08,Strong abnormal issue,Packaging complaints appear 4.1× more often th...
4,Acne Solutions All-Over Clearing Treatment Oil...,CLINIQUE,4.0182,713.0,breakouts,3.86,Strong abnormal issue,Breakouts complaints appear 3.9× more often th...
21,Caffeine 5% + EGCG Depuffing Eye Serum,The Ordinary,3.7715,2118.0,ineffective,3.73,Strong abnormal issue,Ineffective complaints appear 3.7× more often ...
39,Drying Lotion,Mario Badescu,4.0955,398.0,breakouts,3.69,Strong abnormal issue,Breakouts complaints appear 3.7× more often th...
111,The Cult Classic Purifying Face Cleanser,TULA Skincare,3.9064,406.0,smell,3.64,Strong abnormal issue,Smell complaints appear 3.6× more often than t...


In [34]:
diagnostic_summary.to_csv(
    "../data/processed/product_diagnostics.csv",
    index=False
)

In [35]:
pd.read_csv(
    "../data/processed/product_diagnostics.csv"
).head()

,product_name,brand_name,price_usd,rating,reviews,loves_count,theme,review_count,percent_of_reviews_product,percent_of_reviews_baseline,lift,issue_strength,insight
0,The Silk Sunscreen Mineral Broad Spectrum SPF ...,Tatcha,62.0,3.3975,483.0,37047,packaging,116,78.91,10.16,7.77,Very strong abnormal issue,Packaging complaints appear 7.8× more often th...
1,Focuspot Micro Tip Patches,Dr. Jart+,20.0,3.0381,578.0,54955,ineffective,26,11.11,1.75,6.35,Very strong abnormal issue,Ineffective complaints appear 6.3× more often ...
2,Pep-Start Eye Cream,CLINIQUE,32.0,3.6769,1006.0,47677,ineffective,21,9.21,1.75,5.26,Very strong abnormal issue,Ineffective complaints appear 5.3× more often ...
3,Super Amino Gel Cleanser,Summer Fridays,35.0,4.0833,312.0,30871,packaging,24,46.15,10.16,4.54,Strong abnormal issue,Packaging complaints appear 4.5× more often th...
4,Oil-Absorbing Pore Treatment Strips,Peace Out,19.0,3.9750,1562.0,102327,ineffective,23,7.17,1.75,4.10,Strong abnormal issue,Ineffective complaints appear 4.1× more often ...


In [38]:
diagnostic_summary.to_csv(
    "../data/processed/product_diagnostics.csv",
    index=False
)

In [39]:
pd.read_csv(
    "../data/processed/product_diagnostics.csv"
).head()

,product_name,brand_name,price_usd,rating,reviews,loves_count,theme,review_count,percent_of_reviews_product,percent_of_reviews_baseline,lift,issue_strength,insight
0,The Silk Sunscreen Mineral Broad Spectrum SPF ...,Tatcha,62.0,3.3975,483.0,37047,packaging,116,78.91,10.16,7.77,Very strong abnormal issue,Packaging complaints appear 7.8× more often th...
1,Focuspot Micro Tip Patches,Dr. Jart+,20.0,3.0381,578.0,54955,ineffective,26,11.11,1.75,6.35,Very strong abnormal issue,Ineffective complaints appear 6.3× more often ...
2,Pep-Start Eye Cream,CLINIQUE,32.0,3.6769,1006.0,47677,ineffective,21,9.21,1.75,5.26,Very strong abnormal issue,Ineffective complaints appear 5.3× more often ...
3,Super Amino Gel Cleanser,Summer Fridays,35.0,4.0833,312.0,30871,packaging,24,46.15,10.16,4.54,Strong abnormal issue,Packaging complaints appear 4.5× more often th...
4,Oil-Absorbing Pore Treatment Strips,Peace Out,19.0,3.9750,1562.0,102327,ineffective,23,7.17,1.75,4.10,Strong abnormal issue,Ineffective complaints appear 4.1× more often ...


In [40]:
all_diagnostics_df.to_csv(
    "../data/processed/all_product_diagnostics.csv",
    index=False
)